                    Phase 1 of Competitor's Strategy Analysis AI Agent

Importing API KEY from .env File

In [1]:
import os

from dotenv import load_dotenv

load_dotenv()

True

Making a Simple LLM Call

In [2]:
from huggingface_hub import InferenceClient

Client = InferenceClient(
    provider = "hf-inference",
    api_key=os.getenv("HuggingFace_API_KEY"),
    #token=os.getenv("HuggingFace_API_KEY"),
    #temperature=0.7,
)
repo_id = "HuggingFaceTB/SmolLM3-3B"

llm = Client.chat.completions.create(
    model=repo_id,
    messages=[
        {
            "role": "user",
            "content": "What is the capital of Pakistan?",
        }
    ],
)
Response = llm.choices[0].message
print(Response)

c:\Users\aliak\OneDrive\Desktop\LangChain_Project\venv2\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ChatCompletionOutputMessage(role='assistant', content="<think>\nOkay, the user is asking for the capital of Pakistan. Let me start by recalling what I know about Pakistan's geography and government structure. Pakistan is a country in South Asia, and I remember that Islamabad is its capital. But wait, I should make sure I'm not confusing it with another city. For example, Lahore is a major city in Pakistan, but it's the capital of Punjab province, not the country. Karachi is the largest city and a major economic hub, but again, not the capital. \n\nI think the capital was moved from Karachi to Islamabad in the 1960s or 1970s. Let me check my memory. Yes, in 1960, the government decided to move the capital from Karachi to Islamabad to make it a planned city, which is located in the Islamabad-Rawalpindi area. So the current capital is Islamabad. \n\nWait, but sometimes people might refer to Rawalpindi as part of the capital, but technically, Islamabad is the capital city. The Islamabad Ca

Loading Pdf Files Via Document Loader

In [3]:
from langchain_community.document_loaders import PyPDFLoader   

data_file = "data"

#only load pdf files
for file in os.listdir(data_file):
    if file.endswith(".pdf"):
        file_path = os.path.join(data_file, file)
        loader = PyPDFLoader(file_path)
        documents = loader.load()
        print(f"Loaded {len(documents)} documents from {file}")
    else:
        print(f"Skipped {file}, not a PDF file.")

Loaded 12 documents from DigitalRightsFoundation.pdf
Loaded 41 documents from National AI Policy Consultation Draft V1.pdf


Splitting Files into Chunks

In [4]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=45,
    length_function=len,
    add_start_index=True,
)

#It uses the split_documents method to split the loaded documents into smaller chunks based on the specified parameters.
chunks = text_splitter.split_documents(documents)
print(len(chunks), "chunks created")

812 chunks created


Converting Chunks into Embeddings using HuggingFaceEmbeddings 

In [1]:

from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L3-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs = {"normalize_embeddings": True}

)


c:\Users\aliak\OneDrive\Desktop\LangChain_Project\venv2\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
No sentence-transformers model found with name sentence-transformers/all-MiniLM-L3-v2. Creating a new one with mean pooling.


OSError: sentence-transformers/all-MiniLM-L3-v2 is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `hf auth login` or by passing `token=<your_token>`

Assembling these Embeddings and Storing in Vector DB(FAISS or Chroma DB)

In [ ]:
from langchain_community.vectorstores import FAISS

vectorstore = None
for i in range(0, len(docs), 50):  # process in chunks of 50 docs
    chunk_store = FAISS.from_documents(docs[i:i+50], embeddings)
    if vectorstore:
        vectorstore.merge_from(chunk_store)
    else:
        vectorstore = chunk_store
#To prevent MemoryError, we process the documents in smaller chunks of 50 at a time.

Linking RAG PipeLine with QARetrieval & HuggingFacePipeline with Chains

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace

# 1. Load tokenizer and model
model_id = "Qwen/Qwen2-1.5B-Instruc"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id)

# 2. Create HF pipeline
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    temperature=0.7,
    max_new_tokens=512
)

# 3. Wrap pipeline for LangChain
hf_pipeline = HuggingFacePipeline(pipeline=pipe)

# 4. Wrap in ChatHuggingFace
chat = ChatHuggingFace(llm=hf_pipeline)

# 5. Test
response = chat.invoke("What is AI Policy of Pakistan focused on and what are digital right of citizens in pakistan?")
print(response.content)


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]c:\Users\aliak\OneDrive\Desktop\LangChain_Project\venv2\Lib\site-packages\huggingface_hub\file_download.py:801: UserWarning: Not enough free disk space to download the file. The expected file size is: 4966.32 MB. The target location C:\Users\aliak\.cache\huggingface\hub\models--HuggingFaceTB--SmolLM3-3B\blobs only has 4108.91 MB free disk space.
  warnings.warn(


: 

The code you posted does **not** have an indentation error in the shown cell. However, if you are seeing an "Expected indented block" error, it usually means that somewhere in your notebook, a function, loop, or conditional statement is missing an indented body.

**Common causes:**
- A function or loop is declared but has no code inside, or the code is not indented.
- A cell was run with incomplete code.

**How to fix:**
1. **Check for missing indentation** after statements like `for`, `if`, `def`, `class`, etc.
2. **Example of correct indentation:**
    ````python
    for i in range(5):
        print(i)  # This line is indented
    ````

3. **If you want an empty block**, use `pass`:
    ````python
    for i in range(5):
        pass
    ````

**In your notebook:**  
- Review all code cells above and below the error for missing or incorrect indentation.
- If you recently edited a cell, make sure all blocks are properly indented.

If you share the exact error message and the cell where it occurs, I can give a more specific fix.

The code you posted does **not** have an indentation error in the shown cell. However, if you are seeing an "Expected indented block" error, it usually means that somewhere in your notebook, a function, loop, or conditional statement is missing an indented body.

**Common causes:**
- A function or loop is declared but has no code inside, or the code is not indented.
- A cell was run with incomplete code.

**How to fix:**
1. **Check for missing indentation** after statements like `for`, `if`, `def`, `class`, etc.
2. **Example of correct indentation:**
    ````python
    for i in range(5):
        print(i)  # This line is indented
    ````

3. **If you want an empty block**, use `pass`:
    ````python
    for i in range(5):
        pass
    ````

**In your notebook:**  
- Review all code cells above and below the error for missing or incorrect indentation.
- If you recently edited a cell, make sure all blocks are properly indented.

If you share the exact error message and the cell where it occurs, I can give a more specific fix.